## Democracy (V-Dem, varieties of democracy)

In [ ]:
pip install rpy2

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
install.packages("IRkernel")
IRkernel::installspec(user = TRUE)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/IRkernel_1.3.2.tar.gz'
Content type 'application/x-gzip' length 45172 bytes (44 KB)
downloaded 44 KB


The downloaded source packages are in
	‘/tmp/RtmpmRi7kt/downloaded_packages’


In [ ]:
%%R
devtools::install_github("vdeminstitute/vdemdata")

library(vdemdata)
library(dplyr)

ℹ The package "remotes" is required.
✖ Would you like to install it?

1: Yes
2: No

Selection: 1

 Found  1  deps for  0/1  pkgs [⠋] Resolving standard (CRAN/BioC) packages
 Found  1  deps for  0/1  pkgs [⠙] Resolving standard (CRAN/BioC) packages
Checking for 8 new metadata files
 Found  1  deps for  0/1  pkgs [⠹] Resolving standard (CRAN/BioC) packages
⠋ Updating metadata database [0/8] | Downloading [0 B / 0 B]
 Found  1  deps for  0/1  pkgs [⠸] Resolving standard (CRAN/BioC) packages
 Found  1  deps for  0/1  pkgs [⠼] Resolving standard (CRAN/BioC) packages
⠙ Updating metadata database [0/8] | Downloading [0 B / 0 B]
 Found  1  deps for  0/1  pkgs [⠴] Resolving standard (CRAN/BioC) packages
⠹ Updating metadata database [0/8] | Downloading [5.36 kB / 5.36 kB]
 Found  1  deps for  0/1  pkgs [⠦] Resolving standard (CRAN/BioC) packages
⠸ Updating metadata database [0/8] | Downloading [208.46 kB / 3.31 MB]
 Found  1  deps for  0/1  pkgs [⠧] Resolving standard (CRAN/BioC) packages
⠼ Upda

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



In [ ]:
%%R
us_democracy_2010_2025 <- vdem %>%
  filter(
    country_name == "United States of America",
    year >= 2010,
    year <= 2027
  ) %>%
  select(
    country_name,
    year,
    v2x_libdem,

    v2x_delibdem
  )

print(us_democracy_2010_2025)


               country_name year v2x_libdem v2x_delibdem
1  United States of America 2010      0.848        0.859
2  United States of America 2011      0.845        0.857
3  United States of America 2012      0.842        0.856
4  United States of America 2013      0.852        0.865
5  United States of America 2014      0.848        0.860
6  United States of America 2015      0.848        0.854
7  United States of America 2016      0.830        0.812
8  United States of America 2017      0.749        0.625
9  United States of America 2018      0.739        0.625
10 United States of America 2019      0.740        0.623
11 United States of America 2020      0.734        0.587
12 United States of America 2021      0.756        0.722
13 United States of America 2022      0.769        0.737
14 United States of America 2023      0.789        0.752
15 United States of America 2024      0.751        0.745
16 United States of America 2025      0.571        0.481


## Press Freedom

In [ ]:
import pandas as pd

SCORE_URL = "https://data360files.worldbank.org/data360-data/data/RWB_PFI/RWB_PFI_SCORE.csv"
RANK_URL  = "https://data360files.worldbank.org/data360-data/data/RWB_PFI/RWB_PFI_RANK.csv"

def _detect_col(cols, candidates):
    cols_l = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cols_l:
            return cols_l[cand.lower()]
    return None

def load_data360_csv(url: str) -> pd.DataFrame:
    # Data360 CSVs sometimes have lots of metadata columns; pandas can still read them directly.
    return pd.read_csv(url)

def extract_country_year_value(df: pd.DataFrame, country: str, start_year: int, end_year: int) -> pd.DataFrame:
    """
    Returns a tidy df with columns: year, value for the requested country.
    Accepts:
      - ISO3 like 'USA' OR country name like 'United States'
    """

    # Common Data360/SDMX patterns
    ctry_code_col = _detect_col(df.columns, ["REF_AREA", "Country Code", "country_code", "Economy Code"])
    ctry_name_col = _detect_col(df.columns, ["REF_AREA_LABEL", "Country Name", "country_name", "Economy"])
    year_col      = _detect_col(df.columns, ["TIME_PERIOD", "Year", "year", "time"])
    value_col     = _detect_col(df.columns, ["OBS_VALUE", "Value", "value", "obs_value"])

    if year_col and value_col and (ctry_code_col or ctry_name_col):
        # Long format (best case)
        sub = df.copy()

        # Filter by country
        if ctry_code_col and len(country) in (2, 3):  # US / USA style
            sub = sub[sub[ctry_code_col].astype(str).str.upper() == country.upper()]
        elif ctry_name_col:
            sub = sub[sub[ctry_name_col].astype(str).str.lower() == country.lower()]
        else:
            # fallback: try name match even if user gave code
            if ctry_name_col:
                sub = sub[sub[ctry_name_col].astype(str).str.lower() == country.lower()]

        if sub.empty:
            # Try alternative match strategy (contains)
            if ctry_name_col:
                sub = df[df[ctry_name_col].astype(str).str.lower().str.contains(country.lower(), na=False)]

        if sub.empty:
            raise ValueError(f"No rows found for country='{country}'. Try ISO3 (e.g., USA) or exact country name.")

        # Coerce year, filter range
        sub = sub[[year_col, value_col]].copy()
        sub[year_col] = pd.to_numeric(sub[year_col], errors="coerce")
        sub = sub.dropna(subset=[year_col])
        sub = sub[(sub[year_col] >= start_year) & (sub[year_col] <= end_year)]

        out = sub.rename(columns={year_col: "year", value_col: "value"}).sort_values("year")
        return out.reset_index(drop=True)

    # Wide format fallback (years as columns)
    year_cols = [str(y) for y in range(start_year, end_year + 1) if str(y) in df.columns]
    if not year_cols:
        raise ValueError("Could not detect year/value columns OR year columns not present in this file.")

    # Try to filter by ISO3/name columns
    if ctry_code_col:
        sub = df[df[ctry_code_col].astype(str).str.upper() == country.upper()]
    elif ctry_name_col:
        sub = df[df[ctry_name_col].astype(str).str.lower() == country.lower()]
    else:
        raise ValueError("Could not detect a country column in the dataset.")

    if sub.empty:
        raise ValueError(f"No rows found for country='{country}' in wide format file.")

    row = sub.iloc[0]
    return pd.DataFrame({"year": [int(y) for y in year_cols],
                         "value": [row[y] for y in year_cols]})

def fetch_rsf_press_freedom(country: str, start_year: int, end_year: int) -> pd.DataFrame:
    score_df = load_data360_csv(SCORE_URL)
    rank_df  = load_data360_csv(RANK_URL)

    score = extract_country_year_value(score_df, country, start_year, end_year).rename(columns={"value": "score"})
    rank  = extract_country_year_value(rank_df,  country, start_year, end_year).rename(columns={"value": "rank"})

    merged = pd.merge(score, rank, on="year", how="outer").sort_values("year")
    return merged.reset_index(drop=True)



In [ ]:
fetch_rsf_press_freedom('United Kingdom',2010,2025)

,year,score,rank
0,2010,6.00,19.0
1,2012,2.00,28.0
2,2013,83.11,29.0
3,2014,80.07,33.0
4,2015,80.00,34.0
5,2016,78.30,38.0
6,2017,77.74,40.0
7,2018,76.75,40.0
8,2019,77.77,33.0
9,2020,77.07,35.0


## iii. Internet freedom (Freedom on the Net)

In [ ]:
import re
import requests
from bs4 import BeautifulSoup

# year = input("Year (e.g., 2023): ").strip()
# country = input("Country name with hyphens (e.g., united-states): ").strip()

def get_freedom_score(year, country):
  URL = f"https://freedomhouse.org/country/{country}/freedom-net/{year}"
  HEADERS = {"User-Agent": "Mozilla/5.0"}

  response = requests.get(URL, headers=HEADERS, timeout=30)

  if response.status_code != 200:
      raise ValueError(f"Page not found: {URL}")

  html = response.text
  soup = BeautifulSoup(html, "html.parser")

  text = soup.get_text(" ", strip=True)

  # Look for pattern like "76 100"
  match = re.search(r"\b(\d{1,3})\s*/?\s*100\b", text)

  if not match:
      raise ValueError("Could not find Freedom on the Net score.")

  score = int(match.group(1))

  print(f"Freedom on the Net {year} ({country.replace('-', ' ').title()}): {score}/100")
  return score

In [ ]:
scores = []
country = "united-kingdom"
for i in range(2016,2026):
  scores.append(get_freedom_score(i,country))

print(scores)


Freedom on the Net 2016 (United Kingdom): 77/100
Freedom on the Net 2017 (United Kingdom): 76/100
Freedom on the Net 2018 (United Kingdom): 77/100
Freedom on the Net 2019 (United Kingdom): 77/100
Freedom on the Net 2020 (United Kingdom): 78/100
Freedom on the Net 2021 (United Kingdom): 78/100
Freedom on the Net 2022 (United Kingdom): 79/100
Freedom on the Net 2023 (United Kingdom): 79/100
Freedom on the Net 2024 (United Kingdom): 78/100
Freedom on the Net 2025 (United Kingdom): 76/100
[77, 76, 77, 77, 78, 78, 79, 79, 78, 76]


## Governance quality (WGI, world governance indicators)

In [ ]:
import requests

def fetch_wgi(country_iso3: str, indicator: str, start_year: int, end_year: int):
    url = f"https://api.worldbank.org/v2/country/{country_iso3}/indicator/{indicator}"
    params = {
        "format": "json",
        "per_page": 20000,
        "date": f"{start_year}:{end_year}",
        "source": 3,
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    # data = [metadata, rows]
    rows = data[1] if isinstance(data, list) and len(data) > 1 else []
    out = {}
    for row in rows:
        year = row.get("date")
        val = row.get("value")
        if year is not None:
            out[int(year)] = val  # val can be None if missing

    # Return sorted list of (year, value)
    return [(y, out.get(y)) for y in range(start_year, end_year + 1)]

# Example: USA PV.EST 2015–2025
series = fetch_wgi("USA", "PV.EST", 2015, 2025)

print("USA PV.EST (Political Stability & Absence of Violence/Terrorism: Estimate)")
for y, v in series:
    print(f"{y}: {v}")

USA PV.EST (Political Stability & Absence of Violence/Terrorism: Estimate)
2015: None
2016: None
2017: None
2018: None
2019: None
2020: None
2021: None
2022: None
2023: None
2024: None
2025: None


In [ ]:
import pandas as pd

def wgi_country_series_wide(df: pd.DataFrame, country_code: str, indicator_code: str,
                            start_year: int, end_year: int):
    year_cols = [str(y) for y in range(start_year, end_year + 1) if str(y) in df.columns]
    if not year_cols:
        raise ValueError(f"No year columns found between {start_year} and {end_year}.")

    sub = df[(df["Country Code"] == country_code) & (df["Indicator Code"] == indicator_code)]
    if sub.empty:
        raise ValueError(f"No rows for {country_code=} and {indicator_code=}")

    row = sub.iloc[0]
    out = pd.DataFrame({"year": [int(y) for y in year_cols],
                        "value": [row[y] for y in year_cols]})
    return out

# Point estimate percentile rank (what you already did)
print(wgi_country_series_wide(df, "USA", "PV.PER.RNK", 2015, 2025))

# ✅ Upper bound percentile rank (this is likely what your website shows)
print(wgi_country_series_wide(df, "USA", "PV.PER.RNK.UPPER", 2015, 2025))

# (optional) Lower bound
print(wgi_country_series_wide(df, "USA", "PV.PER.RNK.LOWER", 2015, 2025))

NameError: name 'df' is not defined

## Internet penetration (World Bank)

In [ ]:
import requests

def fetch_internet_usage(country_code, start_year, end_year):
    """
    Uses World Bank API to fetch IT.NET.USER.ZS (Internet users % of population)
    for a given country code and year range.
    """
    url = f"https://api.worldbank.org/v2/country/{country_code}/indicator/IT.NET.USER.ZS"
    params = {
        "format": "json",
        "per_page": 500,
        "date": f"{start_year}:{end_year}"
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    # data is a list: [metadata, observations]
    if not isinstance(data, list) or len(data) < 2:
        raise ValueError("Unexpected World Bank API response.")

    results = []
    for entry in data[1]:
        year = int(entry["date"])
        value = entry["value"]  # may be None if missing
        results.append({"year": year, "internet_users_pct": value})

    results.sort(key=lambda x: x["year"])
    return results

# Example usage:
country = "GB"        # ISO-2 country code for United States
start, end = 2010, 2026

internet_data = fetch_internet_usage(country, start, end)
for row in internet_data:
    print(row["year"], row["internet_users_pct"])

2010 85
2011 85.38
2012 87.48
2013 89.8441
2014 91.6133
2015 92.0003
2016 94.7758
2017 90.4246
2018 90.692
2019 92.5166
2020 94.8182
2021 96.1965
2022 96.2989
2023 96.2988
2024 None
2025 None


## Estimation of social media usage


In [ ]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, UrlContext
import os, re, json, math, tempfile


client = genai.Client(api_key="Insert_Key_Here")

def est(country, year, url):
# 1. Define the tool
  tools = [Tool(url_context=UrlContext())]

  METRICS = [
      "population",
      "internet_users",
      "social_media_users",
      "youtube_users",
      "facebook_users",
      "instagram_users",
      "tiktok_users",
      "linkedin_users",
      "messenger_users",
      "snapchat_users",
      "x_users",
      "pinterest_users",
  ]

  schema = {
          "country": {country},
          "year": {year},
          "population_million": "number|null",
          "population_evidence": "string|null",
          "metrics": {m: {"users_million": "number|null",
                          "percent_population": "number|null",
                          "evidence": "string|null"} for m in METRICS if m != "population"}
      }

  prompt = f"""
  Return ONLY valid JSON (no markdown).
  Extract the following metrics from the link {url} for {year}.
  Do NOT guess. If a value is not explicitly in the link, check the slideshare, set it to null.
  Do not calculate if the population is not a given number, just take the number if reported in thousands or millions
  DO NOT CALCULATE METRICS IF ONLY PERCENTAGE IS GIVEN

  Metrics in Million and percentage as per country's population:
  population (million)
  internet users (million)
  social media users (million)
  youtube users (million)
  facebook users (million)
  instagram users (million)
  tiktok users (million)
  linkedin users (million)
  messenger users (million)
  snapchat users (million)
  x users (million)
  pinterest users (million)

  Output must be only the JSON, such as in the {schema} format.
  Do Not explain or give reasoning, just json output.
  """

  # 2. Pass the URL in your prompt
  response = client.models.generate_content(
      model="gemini-2.5-flash", # or gemini-1.5-pro
      contents=prompt,
      config=GenerateContentConfig(tools=tools)
  )

  text = response.text
  return text

In [ ]:
text = est('Denmark', '2016', 'https://www.slideshare.net/slideshow/digital-2018-united-states-of-america-january-2018/119186576')

In [ ]:
print(text)

```json
{
  "country": "United States",
  "year": 2018,
  "population_million": 325.6,
  "population_evidence": "Slide 8, 10",
  "metrics": {
    "internet_users": {
      "users_million": 286.9,
      "percent_population": 88,
      "evidence": "Slide 8, 14"
    },
    "social_media_users": {
      "users_million": 230.0,
      "percent_population": 71,
      "evidence": "Slide 8, 25"
    },
    "youtube_users": {
      "users_million": null,
      "percent_population": 73,
      "evidence": "Slide 26"
    },
    "facebook_users": {
      "users_million": 230.0,
      "percent_population": 72,
      "evidence": "Slide 27 (users_million), Slide 26 (percent_population)"
    },
    "instagram_users": {
      "users_million": 110.0,
      "percent_population": 34,
      "evidence": "Slide 31"
    },
    "tiktok_users": {
      "users_million": null,
      "percent_population": null,
      "evidence": null
    },
    "linkedin_users": {
      "users_million": null,
      "percent_populatio

In [ ]:
def combine(all,data):
  all[str(int(data['year']))].append(data['population_million'])
  all[str(int(data['year']))].append([data['metrics']['internet_users']['users_million'],data['metrics']['internet_users']['percent_population']])

  all[str(int(data['year']))].append([data['metrics']['social_media_users']['users_million'],data['metrics']['social_media_users']['percent_population']])

  #youtube users

  all[str(int(data['year']))].append([data['metrics']['youtube_users']['users_million'],data['metrics']['youtube_users']['percent_population']])

  #facebook users
  all[str(int(data['year']))].append([data['metrics']['facebook_users']['users_million'],data['metrics']['facebook_users']['percent_population']])

  #instagram users
  all[str(int(data['year']))].append([data['metrics']['instagram_users']['users_million'],data['metrics']['instagram_users']['percent_population']])

  #x_users users
  all[str(int(data['year']))].append([data['metrics']['x_users']['users_million'],data['metrics']['x_users']['percent_population']])

  #twitter users

  all[str(int(data['year']))].append([data['metrics']['tiktok_users']['users_million'],data['metrics']['tiktok_users']['percent_population']])


  #linkedin users
  all[str(int(data['year']))].append([data['metrics']['linkedin_users']['users_million'],data['metrics']['linkedin_users']['percent_population']])

  #messenger users
  all[str(int(data['year']))].append([data['metrics']['messenger_users']['users_million'],data['metrics']['messenger_users']['percent_population']])

  #snapchat users

  all[str(int(data['year']))].append([data['metrics']['snapchat_users']['users_million'],data['metrics']['snapchat_users']['percent_population']])


  #pinterest users
  all[str(int(data['year']))].append([data['metrics']['pinterest_users']['users_million'],data['metrics']['pinterest_users']['percent_population']])

  return all


In [ ]:
country = input("Enter country name: ")
country_iso2 = input("Enter country ISO-2 code: ")
min_year = int(input("Enter minimum year: "))
max_year = int(input("Enter maximum year: "))

if country == 'United States':
  country_democracy = 'United States of America'

Enter country name: United States
Enter country ISO-2 code: US
Enter minimum year: 2024
Enter maximum year: 2027


In [ ]:
%%R -i country_democracy -i min_year -i max_year -o democracy_df

library(dplyr)

democracy_df <- vdem %>%
  filter(
    country_name == country_democracy,
    year >= min_year,
    year <= max_year
  ) %>%
  select(
    country_name,
    year,
    v2x_libdem,
    v2x_delibdem
  )

democracy_df

              country_name year v2x_libdem v2x_delibdem
1 United States of America 2024      0.751        0.745
2 United States of America 2025      0.571        0.481


In [ ]:
all = {}
#define the country
all['country'] = country

#Get Democracy and Polarization
for i in range(min_year,max_year+1):
  all[str(int(i))] = []
  if i not in democracy_df['year'].astype('int').tolist():
    all[str(int(i))].append(None)
    all[str(int(i))].append(None)

for i in democracy_df.itertuples(index=False):
  all[str(int(i.year))].append(i.v2x_libdem)
  all[str(int(i.year))].append(i.v2x_delibdem)

country_clean = country.lower().replace(' ', '-')

#get internet score
for i in range(min_year,max_year+1):
  try:
    if i <2016 or i > 2026:
      all[str(int(i))].append(None)
    else:
      all[str(int(i))].append(get_freedom_score(int(i),country_clean))
  except:
      all[str(int(i))].append(None)


#Get internet penetration
internet_data = fetch_internet_usage(country_iso2, min_year, max_year)
internet_data_years = []
for row in internet_data:
  if row['internet_users_pct'] != None:
    all[str(int(row['year']))].append(round(row['internet_users_pct']))
  else:
    all[str(int(row['year']))].append((row['internet_users_pct']))

  internet_data_years.append(row['year'])

for i in range(min_year,max_year+1):
  if i not in internet_data_years:
    all[str(int(i))].append(None)

#Get press freedom
df_press = fetch_rsf_press_freedom(country, min_year, max_year)

for row in df_press.itertuples(index=False):
  all[str(int(row.year))].append(row.score)


for i in range(min_year,max_year+1):
  if i not in df_press['year'].astype('int').tolist():
    all[str(int(i))].append(None)

print(all)


Freedom on the Net 2024 (United States): 76/100
Freedom on the Net 2025 (United States): 73/100
{'country': 'United States', '2024': [0.751, 0.745, 76, None, 66.59], '2025': [0.571, 0.481, 73, None, 65.487], '2026': [None, None, None, None, None], '2027': [None, None, None, None, None]}


In [ ]:
import time

final = all

links = ['https://datareportal.com/reports/digital-2026-united-states-of-america?rq=united%20states'
]

for i in links:
  match = re.search(r"\b(20\d{2})\b", i)

  year = int(match.group()) if match else None
  if len(final[str(year)]) == len(all[str(year)]):
    text = est(country, year, i)
    if not text:
        raise ValueError("Empty response from Gemini")

    # Extract JSON object safely
    import re
    match = re.search(r"\{.*\}", text, re.DOTALL)

    if not match:
        raise ValueError("No JSON found in response")

    clean_json = match.group()
    print('finished', i)
    data = json.loads(clean_json)
    final = combine(all,data)
  #time.sleep(10)


finished https://datareportal.com/reports/digital-2026-united-states-of-america?rq=united%20states


In [ ]:
print(final)
longest_key = max(final.values(), key=len)
print(len(longest_key))

for i in final:
  if len(final[i]) < len(longest_key) and i != 'country':
    final[i] = final[i] + [None] * (len(longest_key) - len(final[i]))
  elif i == 'country' and len(country) != len(longest_key):
    final[i] = [country] * len(longest_key)
print(final)


{'country': 'United States', '2024': [0.751, 0.745, 76, None, 66.59], '2025': [0.571, 0.481, 73, None, 65.487], '2026': [None, None, None, None, None, 348, [324, 93.1], [254, 73.0], [254, 73.0], [198, 56.8], [182, 52.3], [99.0, 28.5], [153, 44.0], [270, 77.6], [6.2, 1.8], [104, 29.8], [96.9, 27.9]], '2027': [None, None, None, None, None]}
17
{'country': ['United States', 'United States', 'United States', 'United States', 'United States', 'United States', 'United States', 'United States', 'United States', 'United States', 'United States', 'United States', 'United States', 'United States', 'United States', 'United States', 'United States'], '2024': [0.751, 0.745, 76, None, 66.59, None, None, None, None, None, None, None, None, None, None, None, None], '2025': [0.571, 0.481, 73, None, 65.487, None, None, None, None, None, None, None, None, None, None, None, None], '2026': [None, None, None, None, None, 348, [324, 93.1], [254, 73.0], [254, 73.0], [198, 56.8], [182, 52.3], [99.0, 28.5], [15

In [ ]:
import pandas as pd

values = ['Democracy','Polarization', 'Internet_Freedom','Internet_Penetration', 'Press_Freedom',
                    'population (million)',
  'internet users (million)',
  'social media users (million)',
  'youtube users (million)',
  'facebook users (million)',
  'instagram users (million)',
          'x_users (million)',
  'tiktok users (million)',
  'linkedin users (million)',
  'messenger users (million)',
  'snapchat users (million)',
  'pinterest users (million)']
final['values'] = values

df = pd.DataFrame(final)

In [ ]:
last_column_name = df.columns[-1]

new_column_order = [last_column_name] + [col for col in df.columns if col != last_column_name]
df = df[new_column_order]


In [ ]:
country = df['country']

df = df.drop(columns=['country'])

df['country'] = country
last_column_name = df.columns[-1]

new_column_order = [last_column_name] + [col for col in df.columns if col != last_column_name]
df = df[new_column_order]

print(df.head())

         country                values    2024    2025  2026  2027
0  United States             Democracy   0.751   0.571  None  None
1  United States          Polarization   0.745   0.481  None  None
2  United States      Internet_Freedom  76.000  73.000  None  None
3  United States  Internet_Penetration     NaN     NaN  None  None
4  United States         Press_Freedom  66.590  65.487  None  None


In [ ]:
df.to_csv(f"{country_clean}.csv")
